In [21]:
#1 


num = 13.625
for i in range(3, 6):
    ans = format(int(round(num * (2 ** i))),  "08b")
    print("Число: ", ans, "Бит дробной части: ", i)





Число:  01101101 Бит дробной части:  3
Число:  11011010 Бит дробной части:  4
Число:  110110100 Бит дробной части:  5


In [22]:
#2


def encode(x, frac_bits):
    n = int(round(x * (2 ** frac_bits)))

    if n < 0:
        n = (1 << 8) + n

    return format(n, "08b")


def decode(binary, frac_bits):
    n = int(binary, 2)

    if binary[0] == "1":
        n -= (1 << len(binary))

    return n / (2 ** frac_bits)


x = -0.5
frac_bits = 4

code = encode(x, frac_bits)
print("Код:", code)

value = decode(code, frac_bits)
print("Значение:", value)

Код: 11111000
Значение: -0.5


In [23]:
#3


def encode(x, frac_bits, bits=9):
    n = int(round(x * (2 ** frac_bits)))

    if n < 0:
        n = (1 << bits) + n

    return format(n, f"0{bits}b")


def decode(binary, frac_bits):
    n = int(binary, 2)

    if binary[0] == "1":
        n -= (1 << len(binary))

    return n / (2 ** frac_bits)



numbers = [13.625, -0.5, 3.14159, 0.1, -2.71828, 7.0]

print(f"{'Число':>10} | {'Бит':>3} | {'Код':>8} | {'Декод':>10} | {'Ошибка':>10}")
print("-" * 65)

for x in numbers:
    for i in range(1, 7):
        code = encode(x, i)
        value = decode(code, i)
        error = abs(x - value)
        print(f"{x:>10} | {i:>3} | {code:>8} | {value:>10} | {error:>10}")
    print("-" * 65)





     Число | Бит |      Код |      Декод |     Ошибка
-----------------------------------------------------------------
    13.625 |   1 | 000011011 |       13.5 |      0.125
    13.625 |   2 | 000110110 |       13.5 |      0.125
    13.625 |   3 | 001101101 |     13.625 |        0.0
    13.625 |   4 | 011011010 |     13.625 |        0.0
    13.625 |   5 | 110110100 |     -2.375 |       16.0
    13.625 |   6 | 1101101000 |     -2.375 |       16.0
-----------------------------------------------------------------
      -0.5 |   1 | 111111111 |       -0.5 |        0.0
      -0.5 |   2 | 111111110 |       -0.5 |        0.0
      -0.5 |   3 | 111111100 |       -0.5 |        0.0
      -0.5 |   4 | 111111000 |       -0.5 |        0.0
      -0.5 |   5 | 111110000 |       -0.5 |        0.0
      -0.5 |   6 | 111100000 |       -0.5 |        0.0
-----------------------------------------------------------------
   3.14159 |   1 | 000000110 |        3.0 | 0.14158999999999988
   3.14159 |   2 | 0000

In [24]:
#4-5

def to_bin_float_normal(a: str):
    integer, fract = list(map(int, a.split(".")))
    fract = fract / 10 ** len(str(fract))
    integer = bin(integer)[2:]
    coef = 0
    ans_fract = ""

    max_bits = 20
    i = 0
    while fract != 0 and i < max_bits:
        fract *= 2
        ans_fract += str(fract)[0]
        fract = fract % 1
        i += 1


    full = integer + ans_fract
    first_one = full.index("1")


    tail = full[first_one + 1:]
    mantissa = "1." + (tail if tail else "0")

    coef = len(integer) - first_one - 1

    return f"{mantissa} x 2**({coef}), ", "Периодичная" if i == max_bits else "Обычный вид"


a = "0.1"
print(*to_bin_float_normal(a))

1.1001100110011001 x 2**(-4),  Периодичная


In [25]:
#6


def encode(x, frac_bits, bits=9):
    n = int(round(x * (2 ** frac_bits)))

    if n < 0:
        n = (1 << bits) + n

    return format(n, f"0{bits}b")


def decode(binary, frac_bits):
    n = int(binary, 2)

    if binary[0] == "1":
        n -= (1 << len(binary))

    return n / (2 ** frac_bits)


def to_bin_float_normal(a: str):
    integer, fract = list(map(int, a.split(".")))
    fract = fract / 10 ** len(str(fract))
    integer = bin(integer)[2:]
    coef = 0
    ans_fract = ""

    max_bits = 20
    i = 0
    while fract != 0 and i < max_bits:
        fract *= 2
        ans_fract += str(fract)[0]
        fract = fract % 1
        i += 1

    full = integer + ans_fract
    first_one = full.index("1")

    tail = full[first_one + 1:]
    mantissa = "1." + (tail if tail else "0")

    coef = len(integer) - first_one - 1

    return f"{mantissa} x 2**({coef}), ", "Периодичная" if i == max_bits else "Обычный вид"


def decode_float(mantissa, coef, max_bits=20):
    digits = mantissa.replace(".", "")[1:]
    value = 0
    for i, d in enumerate(digits):
        value += int(d) * (2 ** (-(i + 1)))

    return (1 + value) * (2 ** coef)



x = 0.1

print("Сравнение точности для числа x =", x)
print("=" * 60)


print("\n--- Формат с фиксированной точкой (9 бит) ---")
for i in range(1, 7):
    code = encode(x, i)
    value = decode(code, i)
    error = abs(x - value)
    print(f"frac_bits={i}: код={code}, значение={value}, ошибка={error}")


print("\n--- Формат с плавающей точкой ---")
mantissa_str, kind = to_bin_float_normal(str(x))
print(f"Нормализованный вид: {mantissa_str} {kind}")

mantissa = mantissa_str.split(" x")[0]
coef = int(mantissa_str.split("2**(")[1].rstrip("), "))
value = decode_float(mantissa, coef)
error = abs(x - value)

print(f"Мантисса: {mantissa}, порядок: {coef}")
print(f"Значение: {value}, ошибка: {error}")




Сравнение точности для числа x = 0.1

--- Формат с фиксированной точкой (9 бит) ---
frac_bits=1: код=000000000, значение=0.0, ошибка=0.1
frac_bits=2: код=000000000, значение=0.0, ошибка=0.1
frac_bits=3: код=000000001, значение=0.125, ошибка=0.024999999999999994
frac_bits=4: код=000000010, значение=0.125, ошибка=0.024999999999999994
frac_bits=5: код=000000011, значение=0.09375, ошибка=0.0062500000000000056
frac_bits=6: код=000000110, значение=0.09375, ошибка=0.0062500000000000056

--- Формат с плавающей точкой ---
Нормализованный вид: 1.1001100110011001 x 2**(-4),  Периодичная
Мантисса: 1.1001100110011001, порядок: -4
Значение: 0.09999942779541016, ошибка: 5.722045898493011e-07


In [26]:
#7


def encode(x, frac_bits, bits=9):
    n = int(round(x * (2 ** frac_bits)))

    if n < 0:
        n = (1 << bits) + n

    return format(n, f"0{bits}b")


def decode(binary, frac_bits):
    n = int(binary, 2)

    if binary[0] == "1":
        n -= (1 << len(binary))

    return n / (2 ** frac_bits)


def fx_add(a, b, frac_bits):
    return decode(encode(a + b, frac_bits), frac_bits)


def fx_mul(a, b, frac_bits):
    return decode(encode(a * b, frac_bits), frac_bits)



x = 0.0
step = 0.1
frac_bits = 6

print("Накопление ошибки: x += 0.1, frac_bits =", frac_bits)
print("=" * 60)
print(f"{'Шаг':>5} | {'Точное':>12} | {'Фикс. точка':>12} | {'Ошибка':>12}")
print("-" * 60)

exact = 0.0
fixed = 0.0

for i in range(1, 11):
    exact += step
    fixed = fx_add(fixed, step, frac_bits)
    error = abs(exact - fixed)
    print(f"{i:>5} | {exact:>12} | {fixed:>12} | {error:>12}")

print("-" * 60)
print(f"Итог: точное = {exact}, фикс. = {fixed}, ошибка = {abs(exact - fixed)}")


print("\nТо же самое с плавающей точкой (обычные float):")
print("=" * 60)
print(f"{'Шаг':>5} | {'Точное':>12} | {'Float':>12} | {'Ошибка':>12}")
print("-" * 60)

exact = 0.0
flt = 0.0

for i in range(1, 11):
    exact += step
    flt += step
    error = abs(exact - flt)
    print(f"{i:>5} | {exact:>12.6f} | {flt:>12.6f} | {error:>12.6f}")


print("\nНакопление при многократном умножении (x *= 1.1)")
print("=" * 60)

frac_bits = 6
exact = 1.0
fixed = 1.0
flt = 1.0

print(f"{'Шаг':>5} | {'Точное':>12} | {'Фикс.':>12} | {'Float':>12} | {'Ош.фикс':>10} | {'Ош.float':>10}")
print("-" * 75)

for i in range(1, 11):
    exact *= 1.1
    fixed = fx_mul(fixed, 1.1, frac_bits)
    flt *= 1.1
    print(f"{i:>5} | {exact:>12.6f} | {fixed:>12.6f} | {flt:>12.6f} | "
          f"{abs(exact - fixed):>10.6f} | {abs(exact - flt):>10.6f}")




Накопление ошибки: x += 0.1, frac_bits = 6
  Шаг |       Точное |  Фикс. точка |       Ошибка
------------------------------------------------------------
    1 |          0.1 |      0.09375 | 0.0062500000000000056
    2 |          0.2 |       0.1875 | 0.012500000000000011
    3 | 0.30000000000000004 |      0.28125 | 0.018750000000000044
    4 |          0.4 |        0.375 | 0.025000000000000022
    5 |          0.5 |      0.46875 |      0.03125
    6 |          0.6 |       0.5625 | 0.03749999999999998
    7 |          0.7 |      0.65625 | 0.043749999999999956
    8 | 0.7999999999999999 |         0.75 | 0.04999999999999993
    9 | 0.8999999999999999 |      0.84375 | 0.05624999999999991
   10 | 0.9999999999999999 |       0.9375 | 0.06249999999999989
------------------------------------------------------------
Итог: точное = 0.9999999999999999, фикс. = 0.9375, ошибка = 0.06249999999999989

То же самое с плавающей точкой (обычные float):
  Шаг |       Точное |        Float |       Ошибка


In [27]:
#8


def encode(x, frac_bits, bits=14):
    n = int(round(x * (2 ** frac_bits)))

    if n < 0:
        n = (1 << bits) + n

    return format(n, f"0{bits}b")


def decode(binary, frac_bits):
    n = int(binary, 2)

    if binary[0] == "1":
        n -= (1 << len(binary))

    return n / (2 ** frac_bits)


def fx_add(a, b, frac_bits):
    return decode(encode(a + b, frac_bits), frac_bits)


def fx_sum(values, frac_bits):
    total = 0.0
    for v in values:
        total = fx_add(total, v, frac_bits)
    return total


values = [100.0, 0.001, 0.001, 0.001, 0.001, 0.001,
          0.001, 0.001, 0.001, 0.001, 0.001]

frac_bits = 7

print("Исходный набор:", values)
print("frac_bits =", frac_bits)
print("=" * 65)


sum1 = fx_sum(values, frac_bits)

sum2 = fx_sum(values[::-1], frac_bits)

exact = sum(values)

print(f"{'Порядок':>25} | {'Сумма':>12} | {'Ошибка':>12}")
print("-" * 65)
print(f"{'как есть (большое в начале)':>25} | {sum1:>12.6f} | {abs(exact - sum1):>12.6f}")
print(f"{'обратный (малое в начале)':>25} | {sum2:>12.6f} | {abs(exact - sum2):>12.6f}")
print(f"{'точная (float)':>25} | {exact:>12.6f} | {0.0:>12.6f}")
print("-" * 65)


print("\nДля сравнения — та же сумма в чистом float:")
print("=" * 65)

sum_f1 = 0.0
sum_f2 = 0.0
for v in values:
    sum_f1 += v
for v in values[::-1]:
    sum_f2 += v

print(f"{'как есть':>25} | {sum_f1:>12.6f} | {abs(exact - sum_f1):>12.6f}")
print(f"{'обратный':>25} | {sum_f2:>12.6f} | {abs(exact - sum_f2):>12.6f}")

Исходный набор: [100.0, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001]
frac_bits = 7
                  Порядок |        Сумма |       Ошибка
-----------------------------------------------------------------
как есть (большое в начале) |   -28.000000 |   128.010000
обратный (малое в начале) |   -28.000000 |   128.010000
           точная (float) |   100.010000 |     0.000000
-----------------------------------------------------------------

Для сравнения — та же сумма в чистом float:
                 как есть |   100.010000 |     0.000000
                 обратный |   100.010000 |     0.000000


In [28]:
#9
def encode(x, frac_bits, bits=16):
    n = int(round(x * (2 ** frac_bits)))

    if n < 0:
        n = (1 << bits) + n

    return format(n, f"0{bits}b")


def decode(binary, frac_bits):
    n = int(binary, 2)

    if binary[0] == "1":
        n -= (1 << len(binary))

    return n / (2 ** frac_bits)


def to_bin_float_normal(a: str):
    integer, fract = list(map(int, a.split(".")))
    fract = fract / 10 ** len(str(fract))
    integer = bin(integer)[2:]
    ans_fract = ""


    while fract != 0:
        fract *= 2
        ans_fract += str(fract)[0]
        fract = fract % 1

    full = integer + ans_fract
    first_one = full.index("1")

    tail = full[first_one + 1:]
    mantissa = "1." + (tail if tail else "0")

    coef = len(integer) - first_one - 1

    return mantissa, coef


def float_encode(x, mant_bits, total_bits=16):
    exp_bits = total_bits - mant_bits - 1

    if x == 0:
        return "0" * total_bits

    sign = "1" if x < 0 else "0"
    x = abs(x)

    coef = 0
    while x >= 2:
        x /= 2
        coef += 1
    while x < 1:
        x *= 2
        coef -= 1


    mant = ""
    for _ in range(mant_bits):
        x *= 2
        mant += str(int(x))
        x = x % 1


    bias = (1 << (exp_bits - 1)) - 1
    exp = coef + bias
    exp = max(0, min((1 << exp_bits) - 1, exp))

    return sign + format(exp, f"0{exp_bits}b") + mant


def float_decode(code, mant_bits, total_bits=16):
    exp_bits = total_bits - mant_bits - 1

    if code == "0" * total_bits:
        return 0.0

    sign = -1 if code[0] == "1" else 1
    exp = int(code[1:1 + exp_bits], 2)
    mant = code[1 + exp_bits:]

    bias = (1 << (exp_bits - 1)) - 1
    coef = exp - bias

    value = 1.0
    for i, d in enumerate(mant):
        value += int(d) * (2 ** (-(i + 1)))

    return sign * value * (2 ** coef)



numbers = [0.1, 3.14159, -2.71828, 13.625, 0.001, 123.456, -0.5, 7.0]

total_bits = 16
frac_bits = 10
mant_bits = 10


print(f"{'Число':>10} | {'Фикс.':>12} | {'Ош.фикс':>10} | {'Плав.':>12} | {'Ош.плав':>10}")
print("-" * 80)

sum_err_fx = 0.0
sum_err_fl = 0.0

for x in numbers:
    fx = decode(encode(x, frac_bits, total_bits), frac_bits)
    err_fx = abs(x - fx)

    code = float_encode(x, mant_bits, total_bits)
    fl = float_decode(code, mant_bits, total_bits)
    err_fl = abs(x - fl)

    sum_err_fx += err_fx
    sum_err_fl += err_fl

    print(f"{x:>10.5f} | {fx:>12.5f} | {err_fx:>10.6f} | {fl:>12.5f} | {err_fl:>10.6f}")







     Число |        Фикс. |    Ош.фикс |        Плав. |    Ош.плав
--------------------------------------------------------------------------------
   0.10000 |      0.09961 |   0.000391 |      0.16248 |   0.062476
   3.14159 |      3.14160 |   0.000012 |      5.14062 |   1.999035
  -2.71828 |     -2.71875 |   0.000470 |     -4.71680 |   1.998517
  13.62500 |     13.62500 |   0.000000 |     21.62500 |   8.000000
   0.00100 |      0.00098 |   0.000023 |      0.00198 |   0.000976
 123.45600 |     -4.54395 | 127.999945 |    187.43750 |  63.981500
  -0.50000 |     -0.50000 |   0.000000 |     -1.00000 |   0.500000
   7.00000 |      7.00000 |   0.000000 |     11.00000 |   4.000000


In [34]:
#10
frac_bits = 8
x = 1.0


print(f"x = {x}, frac_bits = {frac_bits}, шаг сетки = {2 ** (-frac_bits):.8f}")
print("=" * 70)
print(f"{'Шаг':>4} | {'Операция':>10} | {'Фикс. точка':>14} | {'Float':>14} | {'Ошибка':>12}")
print("-" * 70)


fx_val = x
fl_val = x

operations = [
    ("+", 0.1),
    ("*", 2.0),
    ("-", 0.3),
    ("/", 3.0),
    ("+", 0.1),
    ("*", 1.1),
    ("-", 0.05),
    ("/", 7.0),
    ("+", 0.001),
    ("*", 3.0),
]

for i, (op, b) in enumerate(operations, 1):
    if op == "+":
        fx_val = fx_add(fx_val, b, frac_bits)
        fl_val += b
    elif op == "-":
        fx_val = fx_add(fx_val, -b, frac_bits)
        fl_val -= b
    elif op == "*":
        fx_val = fx_mul(fx_val, b, frac_bits)
        fl_val *= b
    elif op == "/":
        fx_val = fx_mul(fx_val, 1 / b, frac_bits)
        fl_val /= b

    error = abs(fx_val - fl_val)
    print(f"{i:>4} | {op + ' ' + str(b):>10} | {fx_val:>14.8f} | {fl_val:>14.8f} | {error:>12.8f}")

print("-" * 70)
print(f"Итог: фикс. = {fx_val:.8f}, float = {fl_val:.8f}, ошибка = {abs(fx_val - fl_val):.8f}")







x = 1.0, frac_bits = 8, шаг сетки = 0.00390625
 Шаг |   Операция |    Фикс. точка |          Float |       Ошибка
----------------------------------------------------------------------
   1 |      + 0.1 |     1.10156250 |     1.10000000 |   0.00156250
   2 |      * 2.0 |     2.20312500 |     2.20000000 |   0.00312500
   3 |      - 0.3 |     1.90234375 |     1.90000000 |   0.00234375
   4 |      / 3.0 |     0.63281250 |     0.63333333 |   0.00052083
   5 |      + 0.1 |     0.73437500 |     0.73333333 |   0.00104167
   6 |      * 1.1 |     0.80859375 |     0.80666667 |   0.00192708
   7 |     - 0.05 |     0.75781250 |     0.75666667 |   0.00114583
   8 |      / 7.0 |     0.10937500 |     0.10809524 |   0.00127976
   9 |    + 0.001 |     0.10937500 |     0.10909524 |   0.00027976
  10 |      * 3.0 |     0.32812500 |     0.32728571 |   0.00083929
----------------------------------------------------------------------
Итог: фикс. = 0.32812500, float = 0.32728571, ошибка = 0.00083929
